# Puru vs Alexander: full reel on Kaggle's free GPUs
**Settings (right panel) before *Run All*:** Accelerator **GPU T4 x2**, Internet **On**.
Optional: add a dataset containing `narration.wav` and `sentences.json` (from `voice_porus.py`) to put the narration in; without it the reel renders with music, Telugu labels and captions only.
Rendering takes about 3-4 hours. The finished file is `/kaggle/working/puru_reel.mp4` (Output tab).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, glob, subprocess
os.chdir('/kaggle/working')
!git clone -q --depth 1 -b reels/follow-the-sugar https://github.com/Dattaharshithreddy/skillvideo.git sv || (cd sv && git pull -q)
!ls sv/reels/porus

In [ ]:
# Blender 4.2 LTS (GPU rendering with OptiX)
if not os.path.exists('/kaggle/working/blender/blender'):
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O b.tar.xz && tar xf b.tar.xz && mv blender-4.2.3-linux-x64 blender && rm b.tar.xz
!apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 libxxf86vm1 libgl1 ffmpeg > /dev/null
!/kaggle/working/blender/blender -b --version | head -1

In [ ]:
# Node 20 + headless Chrome for the Telugu motion-design layer (Remotion)
!node -v 2>/dev/null | grep -q '^v2[0-9]' || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get -qq install -y nodejs > /dev/null)
!apt-get -qq install -y libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libasound2 > /dev/null
!cd sv/reels/porus/mg && npm install --no-audit --no-fund --loglevel=error && npx remotion browser ensure > /dev/null && node -v

In [ ]:
# narration (optional): any attached dataset with narration.wav + sentences.json
narr = glob.glob('/kaggle/input/**/narration.wav', recursive=True); sents = glob.glob('/kaggle/input/**/sentences.json', recursive=True)
os.environ['NARR'] = narr[0] if narr else ''; os.environ['SENTS'] = sents[0] if sents else ''
print('narration:', os.environ['NARR'] or 'none (music only)')

In [ ]:
# render everything: 12 shots across both T4 GPUs, labels, music, Telugu overlay
os.environ.update(BL='/kaggle/working/blender/blender', NGPU='2', SPP='40', CONC='4')
!bash sv/reels/porus/render_all.sh 2>&1 | grep -vE 'INFO|^\s*$' | tail -40

In [ ]:
!cp sv/reels/porus/mg/out/puru_reel.mp4 /kaggle/working/puru_reel.mp4 && ls -la /kaggle/working/puru_reel.mp4
from IPython.display import Video
Video('/kaggle/working/puru_reel.mp4', embed=False, width=320)